In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
db_path = Path(r"D:\ProyectoAnalisisElectrico\MedidasValorizadas")
to_save_path = Path(r"D:\ProyectoAnalisisElectrico\DiaPromedio\Mensuales")

Lo que voy a obtener sera una base de datos por con 1 entrada para las horas 0 para cada conjunto de clave, tension, barra, zona, razon_social, rut, nombre_corto. y deberian ser 24 entradas en total para cada cual de ese conjunto. 

In [ ]:
agg_rules = {
    'medida_3_mean': 'mean', # Tambien creo que esta bien 
    'CMg[CLP/KWh]': 'mean', # El costo creo que tambien esta bien
    'valorizado_CLP': 'mean', # Si el promedio esta bien
    'Fecha_Medicion': 'last', # me tengo que quedar con el mes
}

group_data = [
    'clave',
    'nombre_barra',
    'tension',
    'Zona',
    'Razon_Social',
    'RUT',
    'Nombre_Corto'
    ]

In [ ]:
file_medidas = db_path / "2601" / "2601_medidas_horarias.parquet"
dfs = []
dfs_problematics = []

df = pd.read_parquet(file_medidas)  

In [11]:
df.columns
df.head()

,Hora,clave,nombre_barra,tension,Zona,Razon_Social,RUT,Nombre_Corto,medida_3_mean,medida_3_min,CMg[CLP/KWh]_mean,valorizado_CLP_mean,Fecha_Medicion_last,HoraDia
0,0,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,-8.575,-9.52,47.405682,-408.736413,2026-01-01 00:45:00,0
1,1,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,-8.190,-8.68,58.531720,-479.347402,2026-01-01 01:45:00,1
2,2,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,-8.050,-8.40,55.392895,-445.536743,2026-01-01 02:45:00,2
3,3,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,-9.100,-9.24,50.556957,-460.086130,2026-01-01 03:45:00,3
4,4,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,-8.260,-8.68,62.562665,-516.048430,2026-01-01 04:45:00,4


In [ ]:
df["HoraDia"] = df["Hora"] % 24 
df["Fecha_Medicion_last"] = pd.to_datetime(df["Fecha_Medicion_last"], format="%Y-%m-%d %H:%M:%S")
df["Año_Mes"] = df["Fecha_Medicion_last"].dt.floor('MS')

In [ ]:
for folder_date in db_path.iterdir():
    if not folder_date.is_dir():
        continue
        
    date = folder_date.name
    
    if int(date) < 2505: # Procesamos solo los datos con el formato nuevo en este codigo
        continue

    # Base de datos antigua
    to_save_folder = to_save_path / f"{date}"
    
    if to_save_folder.is_dir():
        print(f"Los datos de {date} ya fueron procesados. Saltando...")
        continue  
        
    to_save_folder.mkdir(parents=True, exist_ok=True)

    file_medidas = folder_date / f"{date}_medidas_horarias.parquet"
    dfs = []
    dfs_problematics = []

    print(f"Empezando {date} ")

    df = pd.read_parquet(file_medidas)  
    
    df["Hora del dia"]


    groups = df.groupby(by=group_data, sort=False) 
    groups_with_size = groups.size()
    good_groups_mask = (groups_with_size == 4)
    
    df_agregado = groups.agg(agg_rules)
    df_buenos = df_agregado[good_groups_mask].reset_index()
    df_buenos.columns = [f"{col[0]}_{col[1]}" if col[1] else col[0] for col in df_buenos.columns]
    dfs.append(df_buenos)
    
    bad_groups_mask = ~good_groups_mask
    if bad_groups_mask.any(): # Si al menos un grupo falló
        print(f"{folder_date.name} tuvo problemas con {name_medida}")
        problematicos = groups.filter(lambda x: len(x) != 4)
        dfs_problematics.append(problematicos)
            
    if dfs:
        df_final_buenos = pd.concat(dfs, ignore_index=True)
        df_final_buenos.to_parquet(to_save_folder / f"{date}_medidas_horarias.parquet", engine="pyarrow", compression="snappy")
        
    if dfs_problematics:
        df_final_malos = pd.concat(dfs_problematics, ignore_index=True)
        df_final_malos.to_csv(to_save_folder / f"{date}_auditoria_errores_15min.csv", sep=";", index=False, encoding="utf-8")